# Meta-Algorithm for fair classification.
The fairness metrics to be optimized have to specified as "input". Currently we can handle the following fairness metrics:
Statistical Rate, False Positive Rate, True Positive Rate, False Negative Rate, True Negative Rate,
Accuracy Rate, False Discovery Rate, False Omission Rate, Positive Predictive Rate, Negative Predictive Rate.

-----------------------------

The example below considers the cases of False Discovery Parity and Statistical Rate (disparate impact).


In [1]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MaxAbsScaler
from tqdm import tqdm

from aif360.datasets import BinaryLabelDataset
from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.datasets import StuperDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric

from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_adult
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import  load_preproc_data_stuper

from aif360.algorithms.inprocessing import MetaFairClassifier

# np.random.seed(12345)

pip install 'aif360[LawSchoolGPA]'
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted 

## Original Training dataset

In [2]:
# 获取数据集，进行训练集和测试集的划分
dataset_orig = load_preproc_data_stuper()

privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

dataset_orig_train, dataset_orig_test = dataset_orig.split([0.7], shuffle=True)

In [3]:
min_max_scaler = MaxAbsScaler()
dataset_orig_train.features = min_max_scaler.fit_transform(dataset_orig_train.features)
dataset_orig_test.features = min_max_scaler.transform(dataset_orig_test.features)

In [4]:
display(Markdown("#### Training Dataset shape"))
print(dataset_orig_train.features.shape)
display(Markdown("#### Favorable and unfavorable labels"))
print(dataset_orig_train.favorable_label, dataset_orig_train.unfavorable_label)
display(Markdown("#### Protected attribute names"))
print(dataset_orig_train.protected_attribute_names)
display(Markdown("#### Privileged and unprivileged protected attribute values"))
print(dataset_orig_train.privileged_protected_attributes, 
      dataset_orig_train.unprivileged_protected_attributes)
display(Markdown("#### Dataset feature names"))
print(dataset_orig_train.feature_names)

#### Training Dataset shape

(454, 137)


#### Favorable and unfavorable labels

1.0 0.0


#### Protected attribute names

['sex']


#### Privileged and unprivileged protected attribute values

[array([1.])] [array([0.])]


#### Dataset feature names

['sex', 'traveltime', 'studytime', 'schoolsup_encoded', 'famsup_encoded', 'paid_encoded', 'activities_encoded', 'nursery_encoded', 'higher_encoded', 'internet_encoded', 'romantic_encoded', 'school_MS', 'address_U', 'famsize_LE3', 'Pstatus_T', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'age=15', 'age=16', 'age=17', 'age=18', 'age=19', 'age=20', 'age=21', 'age=22', 'Medu=0', 'Medu=1', 'Medu=2', 'Medu=3', 'Medu=4', 'Fedu=0', 'Fedu=1', 'Fedu=2', 'Fedu=3', 'Fedu=4', 'failures=0', 'failures=1', 'failures=2', 'failures=3', 'famrel=1', 'famrel=2', 'famrel=3', 'famrel=4', 'famrel=5', 'freetime=1', 'freetime=2', 'freetime=3', 'freetime=4', 'freetime=5', 'goout=1', 'goout=2', 'goout=3', 'goout=4', 'goout=5', 'Dalc=1', 'Dalc=2', 'Dalc=3', 'Dalc=4', 'Dalc=5', 'Walc=1', 'Walc=2', 'Walc=3', 'Walc=4', 'Walc=5', 'health=1', 'health=2',

In [5]:
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = {:.3f}".format(metric_orig_train.mean_difference()))
metric_orig_test = BinaryLabelDatasetMetric(dataset_orig_test, 
                                            unprivileged_groups=unprivileged_groups,
                                            privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = {:.3f}".format(metric_orig_test.mean_difference()))

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.119
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.191


## Algorithm without debiasing

Get classifier without fairness constraints

In [6]:
biased_model = MetaFairClassifier(tau=0, sensitive_attr="sex", type="fdr").fit(dataset_orig_train)

Apply the unconstrained model to test data

In [7]:
dataset_bias_test = biased_model.predict(dataset_orig_test)

In [8]:
classified_metric_bias_test = ClassificationMetric(dataset_orig_test, dataset_bias_test,
                                                   unprivileged_groups=unprivileged_groups,
                                                   privileged_groups=privileged_groups)
print("Test set: Classification accuracy = {:.3f}".format(classified_metric_bias_test.accuracy()))
TPR = classified_metric_bias_test.true_positive_rate()
TNR = classified_metric_bias_test.true_negative_rate()
bal_acc_bias_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = {:.3f}".format(bal_acc_bias_test))
print("Test set: Disparate impact = {:.3f}".format(classified_metric_bias_test.disparate_impact()))
fdr = classified_metric_bias_test.false_discovery_rate_ratio()
fdr = min(fdr, 1/fdr)
print("Test set: False discovery rate ratio = {:.3f}".format(fdr))

Test set: Classification accuracy = 0.913
Test set: Balanced classification accuracy = 0.911
Test set: Disparate impact = 0.667
Test set: False discovery rate ratio = 0.426


## Debiasing with FDR objective

Learn a debiased classifier

In [9]:
debiased_model = MetaFairClassifier(tau=0.7, sensitive_attr="sex", type="fdr").fit(dataset_orig_train)

Apply the debiased model to test data

In [10]:
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)

### Model - with debiasing - dataset metrics

In [11]:
metric_dataset_debiasing_test = BinaryLabelDatasetMetric(dataset_debiasing_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = {:.3f}".format(metric_dataset_debiasing_test.mean_difference()))

Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.203


### Model - with debiasing - classification metrics

In [12]:
classified_metric_debiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 dataset_debiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)

TPR = classified_metric_debiasing_test.true_positive_rate()
TNR = classified_metric_debiasing_test.true_negative_rate()
bal_acc_debiasing_test = 0.5*(TPR+TNR)

print("Test set: Balanced classification accuracy = {:.3f}".format(bal_acc_debiasing_test))

print("Test set: Statistical parity difference = %f" % classified_metric_debiasing_test.statistical_parity_difference())
print("Test set: Equal opportunity difference = %f" % classified_metric_debiasing_test.equal_opportunity_difference())
print("Test set: Disparate impact = {:.3f}".format(classified_metric_debiasing_test.disparate_impact()))

Test set: Balanced classification accuracy = 0.862
Test set: Statistical parity difference = -0.203339
Test set: Equal opportunity difference = -0.111111
Test set: Disparate impact = 0.694
